<a href="https://colab.research.google.com/github/BODUNOVAsofia/python-ai-bodunova-sofia/blob/main/notebooks/week3_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- `paintings.csv` — данные о картинах (название, размеры, год создания, страна, материал, автор)

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

In [ ]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

if not os.path.exists("python-ai-bodunova-sofia"):
    !git clone -q https://github.com/BODUNOVAsofia/python-ai-bodunova-sofia.git

%cd python-ai-bodunova-sofia

print("✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-bodunova-sofia")

/content/python-ai-bodunova-sofia/python-ai-bodunova-sofia/python-ai-bodunova-sofia
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-bodunova-sofia


In [ ]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas

import pandas as pd

df_paintings = pd.read_csv("data/paintings.csv")

print("✅ Загружено строк в df_paintings:", len(df_paintings))

✅ Загружено строк в df_paintings: 1000


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `painting` с URL (ссылкой на объект Wikidata) — нам не нужна ссылка, нам достаточно названия картины.
- Столбцы `paintingLabel`, `countryLabel`, `materialLabel`, `authorLabel` содержат читаемые подписи (название картины, страна, материал, автор).

В этом шаге мы:
- удалим столбец с URL Wikidata (`painting`);
- переименуем `paintingLabel → painting`, `countryLabel → country`, `materialLabel → material`, `authorLabel → author`;
- приведём числовые столбцы (`height`, `width`, `creationYear`) к типу `int`.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа.

In [ ]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# 1) Удаляем столбец с URL Wikidata (он нам не нужен)
df_paintings = df_paintings.drop(columns=["painting"])

# 2) Переименовываем столбцы, убирая постфикс Label
df_paintings = df_paintings.rename(columns={
    "paintingLabel": "painting",
    "countryLabel": "country",
    "materialLabel": "material",
    "authorLabel": "author"
})

# 3) Приводим числовые столбцы к целочисленному типу (int)
for col in ["height", "width", "creationYear"]:
    df_paintings[col] = pd.to_numeric(
        df_paintings[col], errors="coerce"
    ).fillna(0).astype(int)

print("✅ Данные очищены и готовы к анализу")

✅ Данные очищены и готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор нашего DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по году создания (`creationYear`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро получать всю эту информацию одной командой.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_paintings, "Данные о картинах (df_paintings)")

print("\n📈 Статистика по году создания (creationYear):")
print(df_paintings["creationYear"].describe())


📊 Данные о картинах (df_paintings)
Размер: (1000, 7)
Столбцы: painting, height, width, creationYear, country, material, author

Первые строки:
                          painting  height  width  creationYear  country  \
0                    Зимняя жертва     640   1360          1915   Швеция   
1  Любовь земная и Любовь небесная     118    279          1514   Италия   
2  Любовь земная и Любовь небесная     118    279          1514   Италия   
3           Портрет Гертруды Стайн      99     81          1905      США   
4                  Гентский алтарь       3      5          1432  Бельгия   

          material             author  
0            холст  Карл Улоф Ларссон  
1  масляные краски             Тициан  
2            холст             Тициан  
3  масляные краски      Пабло Пикассо  
4   дубовая панель         Ян ван Эйк  

📈 Статистика по году создания (creationYear):
count    1000.000000
mean     1610.066000
std       163.182076
min      1150.000000
25%      1488.000000
50%    

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** картин, стран, материалов и авторов есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу строк);
- **какие материалы самые популярные** (Топ‑10 по числу строк).

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому  
`df_paintings["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

print("\nУникальных картин:", df_paintings["painting"].nunique())
print("Уникальных стран:", df_paintings["country"].nunique())
print("Уникальных материалов:", df_paintings["material"].nunique())
print("Уникальных авторов:", df_paintings["author"].nunique())

print("\nДиапазон годов создания:")
print(f"  От {df_paintings['creationYear'].min()} до {df_paintings['creationYear'].max()}")

print("\nТоп-5 стран по числу записей:")
print(df_paintings["country"].value_counts().head())

print("\nТоп-10 материалов:")
print(df_paintings["material"].value_counts().head(10))

🔍 Быстрая проверка данных

Уникальных картин: 380
Уникальных стран: 28
Уникальных материалов: 34
Уникальных авторов: 169

Диапазон годов создания:
  От 1150 до 1967

Топ-5 стран по числу записей:
country
Италия      303
Франция     115
Германия     86
Бельгия      80
США          79
Name: count, dtype: int64

Топ-10 материалов:
material
масляные краски     386
холст               233
панель              165
темпера              61
штукатурка           22
fresco               17
дубовая панель       16
дуб                  16
панель из тополя     12
золото               11
Name: count, dtype: int64


## 📏 [5A] Эволюция масштаба: от камерных работ до монументальных полотен

Посчитаем площадь каждой картины (`height × width`) и посмотрим, как менялся средний размер картин от столетия к столетию.

**Что мы делаем:**
- Создаём новый столбец `area` — площадь картины в см²
- Вычисляем век из года создания: `(год - 1) // 100 + 1`
- Группируем картины по векам и считаем среднюю площадь для каждого века
- Находим века с самыми большими и самыми маленькими картинами

**Зачем это нужно:**
Становилось ли искусство со временем более «гигантским» (например, для дворцов и галерей) или, наоборот, уходило в «камерные» форматы?

In [ ]:
# 📏 Идея 1. Эволюция масштаба картин по столетиям

# 1) Создаём столбец с площадью картины (в см²)
df_paintings["area"] = df_paintings["height"] * df_paintings["width"]

# 2) Вычисляем век из года создания
# Формула: (год - 1) // 100 + 1 даёт правильный век (например, 1514 → XVI век)
df_paintings["century"] = (df_paintings["creationYear"] - 1) // 100 + 1

# 3) Считаем среднюю площадь для каждого века
century_stats = df_paintings.groupby("century")["area"].mean().sort_index()

print("📊 Средняя площадь картин по столетиям (см²):")
print(century_stats.round(0).astype(int))

# 4) Дополнительные текстовые выводы
max_century = century_stats.idxmax()
min_century = century_stats.idxmin()

print(f"\n🏆 Век с самыми БОЛЬШИМИ картинами: {max_century} век (средняя площадь {century_stats[max_century]:.0f} см²)")
print(f"🔬 Век с самыми МАЛЕНЬКИМИ картинами: {min_century} век (средняя площадь {century_stats[min_century]:.0f} см²)")
print(f"📐 Разница между ними: {century_stats[max_century] / century_stats[min_century]:.1f} раз(а)")

📊 Средняя площадь картин по столетиям (см²):
century
12    2572400
13      96420
14       5225
15      29032
16      39382
17     109769
18      57496
19     153712
20      45560
Name: area, dtype: int64

🏆 Век с самыми БОЛЬШИМИ картинами: 12 век (средняя площадь 2572400 см²)
🔬 Век с самыми МАЛЕНЬКИМИ картинами: 14 век (средняя площадь 5225 см²)
📐 Разница между ними: 492.3 раз(а)


## 🚀 [5B] Аномалии и выбросы: картины, сломавшие систему

С помощью статистики найдём работы, которые сильно выбиваются из общих трендов своей эпохи.

**Что мы делаем:**

**1. Аномалии по РАЗМЕРУ внутри своего века:**
- Для каждого века считаем среднюю площадь и стандартное отклонение
- Вычисляем Z-score: насколько сильно каждая картина отклоняется от среднего по своему веку
- Находим картины с `|z-score| > 2` — это выбросы за 2 стандартных отклонения

**2. Аномалии по МАТЕРИАЛУ: нетипичные сочетания страна + материал:**
- Считаем, насколько редко встречается данный материал в данной стране
- Находим комбинации, где доля материала составляет менее 2% от всех картин этой страны

**Зачем это нужно:**
Это истории про художников, которые опередили своё время! Например, картина аномально большого размера для XV века или использование нетипичного материала для конкретной страны.

In [ ]:
# 🚀 Идея 2. Поиск аномалий: картины, выбивающиеся из трендов

# 1) Находим аномалии по РАЗМЕРУ внутри своего века
# Для каждого века считаем среднее и стандартное отклонение площади
century_mean = df_paintings.groupby("century")["area"].transform("mean")
century_std = df_paintings.groupby("century")["area"].transform("std")

# Z-score: насколько сильно картина отклоняется от среднего по своему веку
df_paintings["size_zscore"] = (df_paintings["area"] - century_mean) / century_std

# Аномалии — это картины с |z-score| > 2 (выбросы за 2 стандартных отклонения)
size_anomalies = df_paintings[df_paintings["size_zscore"].abs() > 2].copy()
size_anomalies = size_anomalies.sort_values("size_zscore", ascending=False)

print("🔍 Топ-10 самых АНОМАЛЬНЫХ картин по размеру для своей эпохи:")
print(size_anomalies[["painting", "author", "century", "area", "size_zscore"]].head(10))

print(f"\n📊 Всего найдено картин-аномалий по размеру: {len(size_anomalies)} из {len(df_paintings)}")

# 2) Находим аномалии по МАТЕРИАЛУ: нетипичные сочетания страна + материал
# Считаем, насколько редко встречается данный материал в данной стране
country_material = df_paintings.groupby(["country", "material"]).size().reset_index(name="count")
total_by_country = df_paintings.groupby("country").size().reset_index(name="total")

country_material = country_material.merge(total_by_country, on="country")
country_material["share"] = country_material["count"] / country_material["total"]

# Редкие комбинации: доля материала в стране < 2%
rare_materials = country_material[country_material["share"] < 0.02]

print("\n🎨 Топ-10 самых РЕДКИХ сочетаний страна + материал:")
print(rare_materials.sort_values("share").head(10)[["country", "material", "count", "share"]])

🔍 Топ-10 самых АНОМАЛЬНЫХ картин по размеру для своей эпохи:
                               painting             author  century     area  \
216                    Un coin de table  Анри Фантен-Латур       19  3600000   
213                    Un coin de table  Анри Фантен-Латур       19  3600000   
593       Madonna and Child with Saints       Якопо Пальма       16   766800   
729       Madonna and Child with Saints       Якопо Пальма       16   766800   
234             Брак в Кане Галилейской     Паоло Веронезе       16   672938   
229             Брак в Кане Галилейской     Паоло Веронезе       16   672938   
774  Saint Mark Preaching in Alexandria   Джованни Беллини       15   267190   
791  Saint Mark Preaching in Alexandria   Джентиле Беллини       15   267190   
770  Saint Mark Preaching in Alexandria   Джованни Беллини       15   267190   
783  Saint Mark Preaching in Alexandria   Джентиле Беллини       15   267190   

     size_zscore  
216     7.360837  
213     7.360837  
5

## ⏳ [5C] Хронология технологий: восход и закат материалов

Посмотрим, как доля использования разных материалов менялась от века к веку.

**Что мы делаем:**
- Считаем количество картин для каждой комбинации (век, материал)
- Для каждого века вычисляем долю каждого материала
- Берём только топ-5 самых популярных материалов за всю историю
- Создаём сводную таблицу: век × материал (доли в процентах)
- Определяем, какой материал доминировал в каждом столетии

**Зачем это нужно:**
В каком веке холст окончательно победил деревянную панель? Когда появились новые, нетипичные основы (например, медь или картон)? Это настоящий график технологического прогресса в живописи!

In [ ]:
# ⏳ Идея 3. Хронология технологий: как менялись материалы по векам

# 1) Считаем количество картин для каждой комбинации (век, материал)
material_by_century = df_paintings.groupby(["century", "material"]).size().reset_index(name="count")

# 2) Для каждого века считаем общую сумму и вычисляем долю материала
total_per_century = material_by_century.groupby("century")["count"].transform("sum")
material_by_century["share"] = material_by_century["count"] / total_per_century

# 3) Берём только топ-5 самых популярных материалов за всю историю
top_materials = df_paintings["material"].value_counts().head(5).index
df_top = material_by_century[material_by_century["material"].isin(top_materials)]

# 4) Создаём сводную таблицу: век × материал
pivot_data = df_top.pivot(index="century", columns="material", values="share").fillna(0)

print("📊 Доля материалов по столетиям (%):")
print((pivot_data * 100).round(1).astype(str) + "%")

# 5) Текстовые выводы: в каком веке какой материал доминировал
print("\n🏆 Доминирующий материал по столетиям:")
for century in pivot_data.index:
    top_mat = pivot_data.loc[century].idxmax()
    top_share = pivot_data.loc[century].max()
    print(f"  {century} век: {top_mat} ({top_share*100:.1f}%)")

📊 Доля материалов по столетиям (%):
material масляные краски панель темпера  холст штукатурка
century                                                  
12                  0.0%  50.0%   50.0%   0.0%       0.0%
14                  0.0%   0.0%    0.0%  14.3%      14.3%
15                 23.7%  30.9%   17.4%   3.2%       6.6%
16                 44.8%  21.4%    1.6%  18.3%       0.0%
17                 49.4%   7.0%    0.0%  39.2%       0.0%
18                 50.0%   0.0%    0.0%  42.5%       0.0%
19                 49.4%   0.6%    0.6%  48.1%       0.0%
20                 44.2%   0.0%    0.0%  42.3%       0.0%

🏆 Доминирующий материал по столетиям:
  12 век: панель (50.0%)
  14 век: холст (14.3%)
  15 век: панель (30.9%)
  16 век: масляные краски (44.8%)
  17 век: масляные краски (49.4%)
  18 век: масляные краски (50.0%)
  19 век: масляные краски (49.4%)
  20 век: масляные краски (44.2%)


## 📝 Summary / Итоги

**Что мы сделали в этом ноутбуке (Week 2):**

✅ Клонировали репозиторий GitHub в Colab  
✅ Прочитали CSV-файл `paintings.csv` с данными о картинах  
✅ Удалили URL Wikidata и переименовали столбцы (убрали постфикс `*Label`)  
✅ Привели числовые столбцы (`height`, `width`, `creationYear`) к целочисленному типу  
✅ Проверили структуру данных и посмотрели базовую статистику по году создания  
✅ Выполнили быструю валидацию:  
  - количество уникальных картин, стран, материалов и авторов  
  - диапазоны значений  
  - топ стран и материалов по числу записей  

**А также провели углубленный статистический анализ:**

📏 **Эволюция масштаба:** посчитали площадь картин и выяснили, как менялся средний размер полотен от века к веку.  
🚀 **Аномалии и выбросы:** нашли картины, которые «сломали систему» (аномально большие или маленькие для своей эпохи) и самые редкие сочетания стран и материалов.  
⏳ **Хронология технологий:** отследили, как менялась доля использования разных материалов (холст, дерево и др.) с течением времени и какой материал доминировал в каждом столетии.

Теперь у нас есть аккуратные, проверенные таблицы и первые интересные выводы о живописи, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы продолжим использовать эти же данные для:

- более сложного анализа (группировки, фильтрация, поиск скрытых закономерностей),
- построения визуализаций (графики и диаграммы) на основе наших расчетов. 🎨